In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer ,Trainer ,TrainingArguments
from datasets import Dataset
import pandas as pd

# **Import data**

In [ ]:
book_texts = pd.read_csv("book.csv")[0].tolist()
movie_texts = pd.read_csv("movie.csv")[0].tolist()
sport_texts = pd.read_csv("sport.csv")[0].tolist()

label_map = {0: "کتاب",1: "سینما",2: "ورزش"}

texts = book_texts + movie_texts + sport_texts

labels = ([0] * len(book_texts) + [1] * len(movie_texts) + [2] * len(sport_texts))

dataset_ = Dataset.from_dict({"text": texts,"label": labels})

# split

In [2]:
dataset = dataset_.train_test_split(test_size=0.2 ,seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]
eval_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 18
})

# Tokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained("demoversion/bert-fa-base-uncased-haddad-wikinli")

def tokenize(examples):
    return tokenizer(examples["text"] ,truncation=True ,padding="max_length" ,max_length=64)

train_dataset = train_dataset.map(tokenize,batched=True)

eval_dataset = eval_dataset.map(tokenize,batched=True)

Map: 100%|██████████| 18/18 [00:00<00:00, 3643.17 examples/s]


In [4]:
train_dataset = train_dataset.remove_columns(["text"])      # remove text column
eval_dataset = eval_dataset.remove_columns(["text"])

In [5]:
train_dataset.set_format("torch")         # convert to Tensor
eval_dataset.set_format("torch")

# Modeling

In [6]:
model = AutoModelForSequenceClassification.from_pretrained("demoversion/bert-fa-base-uncased-haddad-wikinli" ,num_labels=3 ,ignore_mismatched_sizes=True)
model.classifier

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at demoversion/bert-fa-base-uncased-haddad-wikinli and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Linear(in_features=768, out_features=3, bias=True)

# Train

In [9]:
training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True)

trainer = Trainer(model=model,args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset)

In [10]:
trainer.train()

  0%|          | 0/180 [00:00<?, ?it/s]

                                                
 10%|█         | 18/180 [00:32<04:25,  1.64s/it]

{'eval_loss': 0.7823622226715088, 'eval_runtime': 0.6649, 'eval_samples_per_second': 27.07, 'eval_steps_per_second': 7.52, 'epoch': 1.0}


                                                
 20%|██        | 36/180 [01:07<03:51,  1.61s/it]

{'eval_loss': 0.420949786901474, 'eval_runtime': 0.6459, 'eval_samples_per_second': 27.87, 'eval_steps_per_second': 7.742, 'epoch': 2.0}


...

                                                 
100%|██████████| 180/180 [06:47<00:00,  1.60s/it]

{'eval_loss': 0.0022128517739474773, 'eval_runtime': 0.7036, 'eval_samples_per_second': 25.584, 'eval_steps_per_second': 7.107, 'epoch': 10.0}


100%|██████████| 180/180 [07:12<00:00,  2.40s/it]

{'train_runtime': 432.3528, 'train_samples_per_second': 1.665, 'train_steps_per_second': 0.416, 'train_loss': 0.1959122127956814, 'epoch': 10.0}


TrainOutput(global_step=180, training_loss=0.1959122127956814, metrics={'train_runtime': 432.3528, 'train_samples_per_second': 1.665, 'train_steps_per_second': 0.416, 'total_flos': 23680207595520.0, 'train_loss': 0.1959122127956814, 'epoch': 10.0})

# Save Model

In [ ]:
model.save_pretrained("./Model_Classification")
tokenizer.save_pretrained("./Model_Classification")